# Python Bootcamp Day 3:
## Lecture 2: Timeseries & Pandas

We will continue our work with timeseries data by producing some plots based on our reformatted wind speed measurements. We will use these measurements to answer questions like "what month has the strongest average winds" and "how often are winds so weak that they don't spin the turbine".

### Goals for the lesson:
- Learn how to analyze the data 
- Learn how to make some simple plots of the data


Reminder:
<br/><font color='#0F766E'>* Teal text indicates activities that you should complete as a student
<br/><font color='#B91C1C'>* Red text indicates extra challenge questions 


In [ ]:
# Path to shared bootcamp data (repo-root Datasets/)
DATA_FOLDER = "../../Datasets/"

### Import libraries / packages
from datetime import datetime  # for some datetime manipulations
import numpy as np # for data storage and math
import pandas as pd # for data storage and timeseries math
import matplotlib.pyplot as plt # for plotting


In [ ]:
# Set the path to any datasets used
DATA_FOLDER = "../../Datasets/"


# 1. Read in data from previous lesson
We saved our nicely reformatted DataFrame from the previous lesson as a `.pkl` file with a different name, `reformatted_lidar_winds.pkl`. Let's start by loading that into a DataFrame and making sure it looks like we remember.

In [ ]:
# We saved this in last lesson's directory, so we've added that path
df = pd.read_pickle(DATA_FOLDER + "reformatted_lidar_winds.pkl")


After running the command above, take a look at our DataFrame so we remember how it is structured:

In [ ]:
df

# 2. Processing the data

Now that our data is nicely formatted, we can begin to ask questions. For example, what is the average wind speed at every height? To get the average wind speed across all timesteps, run `df.mean()`. This function averages along every **column!**

In [ ]:
df.mean()

Instead of selecting all timestamps, we can also subselect data by characteristics. For example, we can subselect all data in September. Note, this grabs data in *both* September 2019 *and* September 2020. Also note the double equals sign, `==`, which is a logical operator meaning "is equal to", whereas a single equals sign is used to assign variables to values.

In [ ]:
df[df.index.month == 9]

You can also grab data from *only* September 2020.

In [ ]:
df[(df.index.month == 9) & (df.index.year == 2020)]

We can find the average wind speeds in September 2020 by running `.mean()` on the data that we subselect.

In [ ]:
df[(df.index.month == 9) & (df.index.year == 2020)].mean()

<font color='#0F766E'> Is the average wind speed in September 2020 higher or lower than the average over the whole dataset? How much higher/lower is it at 98m? </font>


### Visualizing the data

So far, we have been looking at average wind speed behavior. But how do winds vary over time? Let's plot 138 m winds.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8,3))

ax.plot(df.index, df['wspd138m'])

ax.set_ylabel("Wind Speed [m/s]", fontsize=12)
ax.set_xlabel("Date", fontsize=12)

plt.show()

We can see that winds at any particular time can be as strong as 30 m/s or as weak as 0 m/s. Let's smooth this time series by averaging winds over every week. More information on the `pandas` resample function can be found [here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.resample.html), but in short this function breaks up the data into shorter chunks (in the example below, each chunk is a week long) and then applies some function separately to each chunk (below, the function is `.mean()`).

In [ ]:
weekly_winds = df.resample("W").mean() # note the "W" stands for 'week'

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8,3))

ax.plot(df.index, df['wspd138m'], color='blue') # the original data, in blue
ax.plot(weekly_winds.index, weekly_winds['wspd138m'], color='orange') # the weekly averages, in orange

ax.set_ylabel("Wind Speed [m/s]", fontsize=12)
ax.set_xlabel("Date", fontsize=12)

plt.show()

<font color="#0F766E"> In the original data, we noticed that the wind can be as strong as 30 m/s or as weak as 0 m/s. What are the maximum and minimum **weekly average** wind speeds? </font>


### Bonus: Adding a new column to your Dataframe

What if we want to add a column of the average windspeed at all altitudes? It only requires 1 line of code!

In [ ]:
df['avg_wspd'] = df.mean(axis=1) # axis=0 is mean over all rows, axis=1 is mean over all columns

In [ ]:
df

# 3. Asking science questions

### What month has the strongest average winds?
Now that we've seen how to subset and visualize the data, let's try asking a more complex question. Our analysis task is to get the average wind speed per month. Our visualization task will then be to either plot the output so we can look for the highest peak or compute the maximum value.

To get the average wind speed per month, we're going to use a nifty function called `.groupby()`. Like `resample`, this will let us split the data into chunks and then apply some function to each chunk. But unlike `resample`, `groupby` lets us use chunks that are discontinuous in time. Let's see the difference when we try to get the average wind speed by month.

In [ ]:
resample_monthly_average = df.resample('ME').mean() # 'ME' stands for month end; the label will be the last day of each month
resample_monthly_average


In the output above, notice that we have an average for each month in each year. Instead, what we want is the average for each month over all years, so the November average would be over November of 2019 and 2020 combined. This is what `groupby` allows us to do:

In [ ]:
groupby_monthly_average = df.groupby(df.index.month).mean()
groupby_monthly_average

Now the timestamp has been converted to a number that represents each of the calendar months, and the average per month is an aggregate over all years in the dataset.


<font color='#0F766E'>Plot `groupby_monthly_average`, with timestamp on the x-axis and wspd78m on the y-axis, to see which month has the highest average wind speed. We're using 78 m because most modern wind turbines are around 80 m tall.


<font color='#0F766E'>Based on the plot above, when do you expect to get the most electricity from an 80m wind turbine at this location? What about the least?


<font color='#B91C1C'> Instead of reading values from a plot or table, calculate the month with the highest average wind speed at 78m using `.idxmax`
